In [1]:
import polars as pl
import json
import yaml

with open("../config.copenhagen_sample.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [3]:
try:
    with open(
        f"{config['dirs']['output']}/{config['city']}/partition_stats.json", "r"
    ) as fp:
        stats = json.load(fp)
except FileNotFoundError:
    # runs 5m 15s for Budapest region
    stats = {}
    for i in range(256):
        data = pl.read_parquet(
            f"{config['dirs']['output']}/{config['city']}/partitioned/{i:02x}/pings.parquet"
        )
        stats[i] = {
            "number_of_records": len(data),
            "number_of_devices": len(data["device_id"].unique()),
        }
        del data
    with open(
        f"{config['dirs']['output']}/{config['city']}/partition_stats.json", "w"
    ) as fp:
        json.dump(stats, fp, indent=1)

In [4]:
records = []
for k, v in stats.items():
    records.append([k, v["number_of_records"], v["number_of_devices"]])
stats_df = pl.from_records(
    records,
    schema=["partition", "number_of_records", "number_of_devices"],
    orient="row",
)

In [5]:
stats_df["number_of_records"].sum()

18871132

In [6]:
print(
    stats_df.describe()
    .drop("partition")
    .with_columns(
        pl.col.number_of_records.round().cast(pl.Int64),
        pl.col.number_of_devices.round().cast(pl.Int64),
    )
    .to_pandas()
    .to_markdown(index=False)
)

| statistic   |   number_of_records |   number_of_devices |
|:------------|--------------------:|--------------------:|
| count       |                 256 |                 256 |
| null_count  |                   0 |                   0 |
| mean        |               73715 |                 378 |
| std         |                7571 |                  20 |
| min         |               54323 |                 309 |
| 25%         |               68882 |                 365 |
| 50%         |               73770 |                 377 |
| 75%         |               79128 |                 392 |
| max         |               94241 |                 424 |
